In [ ]:
import sys
sys.path.insert(0, '../')

import pandas as pd

from automed import AutoMed

In [ ]:
titanic = pd.read_csv('../perf_logger/tests_data/titanic.csv', delimiter=';')

In [ ]:
autom = AutoMed()

print(autom.debug_load())
print(autom.json_pipeline())

In [ ]:
pipeline = {
    'step': 'MetaOrderedStep',
    'children': [
        {
            'step': 'MetaStep',
            'tag': 'cleaning'
        },
        {
            'step': 'MetaStep',
            'tag': 'metric'
        },
        {
            'step': 'KFold',
            'children': [{
                'step': 'ActSVM'
            }]
        }
    ]
}

autom.load_pipeline(pipeline)
print(autom.json_pipeline())


In [ ]:
print(titanic[['label']])
results = autom.fit(
    X=titanic.drop('label', axis=1),
    Y=titanic[['label']])
[ (r.model.ml_model, r.evaluate()) for r in results if r.model.ml_model is not None ]

In [ ]:
print(results[0].model.ml_model)
print(results[0].model.stack)

pm = results[0].model.pickle()
[ len(r.model.pickle()) for r in results ]

In [ ]:
import pickle

o = 200 # offset
n = 68  # # of samples
labels = titanic.iloc[o:(o+n)]['label']
predict_df = titanic.iloc[o:(o+n)].drop('label', axis=1).copy()

# labels = labels.reset_index()
predict_df.reset_index(inplace=True, drop=True)

m = pickle.loads(pm)

results = m.predict(predict_df)
sum([ r == labels[o+i] for i, r in enumerate(results) ]) / n

In [ ]:
final_boss_automed = AutoMed(max_workers=12)
final_boss_automed.debug_load()
final_boss_results = final_boss_automed.fit(titanic.drop('label', axis=1).copy(), titanic[['label']].copy())

In [ ]:
# sum([ len(r.model.pickle()) for r in final_boss_results ])
[ (r.model.ml_model, r.evaluate()) for r in final_boss_results if r.model.ml_model is not None ]